# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

In [1]:
import re
import numpy as np
import pandas as pd
import random
import nltk

import gensim.downloader as api
from collections import Counter

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_predict

W1123 16:14:19.803000 22624 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
%pip install numpyro
import sys
from pathlib import Path

root = Path.cwd().parent

sys.path.insert(0, str(root / "part1"))
sys.path.insert(0, str(root))

import part1
import part2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Load dataset
ds = load_dataset("gsri-18/ISEAR-dataset-complete")
df = ds["train"].to_pandas()

# Separate texts and labels
X = df["content"].astype(str).tolist()
y = df["emotion"].tolist()

print("Dataset size:", len(df))
print("\nEmotion distribution:")
print(Counter(y))

Dataset size: 7516

Emotion distribution:
Counter({'joy': 1092, 'sadness': 1082, 'anger': 1079, 'fear': 1076, 'shame': 1071, 'disgust': 1066, 'guilt': 1050})


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, 
                                                    random_state=42, stratify=y)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 5261
Test samples: 2255


# **Find a suitable word embedding model**

### Word2Vec Google News (word-level embeddings)
### Sentence-BERT: all-MiniLM-L6-v2 (sentence-level embeddings)


## word2vec embedding

In [6]:
# load pretrained Word2Vec
word2vec = api.load("word2vec-google-news-300")

print("Vocabulary size:", len(word2vec))
print("Embedding dimensions:", word2vec.vector_size)

# split text into words
def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text)

# get Word2Vec vector
def get_vector(word):
    for candidate in [word, word.lower(), word.capitalize()]:
        if candidate in word2vec:
            return word2vec[candidate]
    return None

tokens = [word for text in X_train for word in tokenize(text)]
unique_words = set(tokens)
covered_words = {word for word in unique_words if get_vector(word) is not None}

print("\nUnique words:", len(unique_words))
print("Covered words:", len(covered_words))
print("Unique word coverage:", round(len(covered_words) / len(unique_words) * 100, 2), "%")

token_coverage = sum(get_vector(word) is not None for word in tokens) / len(tokens)

print("Token coverage:", round(token_coverage * 100, 2), "%")


# Create one embedding per text by averaging word vectors
def word2vec_embedding(text):
    vectors = [get_vector(word) for word in tokenize(text) if get_vector(word) is not None]

    if not vectors:
        return np.zeros(word2vec.vector_size)

    return np.mean(vectors, axis=0)

X_train_word2vec = np.array([word2vec_embedding(text) for text in X_train])
X_test_word2vec = np.array([word2vec_embedding(text) for text in X_test])

print("\nWord2Vec training shape:", X_train_word2vec.shape)
print("Word2Vec test shape:", X_test_word2vec.shape)

Vocabulary size: 3000000
Embedding dimensions: 300

Unique words: 8096
Covered words: 7764
Unique word coverage: 95.9 %
Token coverage: 99.66 %

Word2Vec training shape: (5261, 300)
Word2Vec test shape: (2255, 300)


## MiniLM embeddings

In [7]:
# load pretrained MiniLM
model_name = "sentence-transformers/all-MiniLM-L6-v2"

minilm = SentenceTransformer(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Embedding dimensions:", minilm.get_sentence_embedding_dimension())
print("Maximum sequence length:", minilm.max_seq_length)

token_counts = [len(tokenizer.tokenize(text)) for text in X_train]

print("\nAverage tokens per text:", round(np.mean(token_counts), 2))
print("Maximum tokens:", max(token_counts))

too_long = sum(count + 2 > minilm.max_seq_length for count in token_counts)
print("Texts exceeding maximum length:", too_long)

X_train_minilm = minilm.encode(X_train, batch_size=32, show_progress_bar=True,
                               convert_to_numpy=True)
X_test_minilm = minilm.encode(X_test, batch_size=32, show_progress_bar=True,
                              convert_to_numpy=True)

print("\nMiniLM training shape:", X_train_minilm.shape)
print("MiniLM test shape:", X_test_minilm.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\juliu\AppData\Local\Temp\ipykernel_22624\651610732.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", minilm.get_sentence_embedding_dimension())


Embedding dimensions: 384
Maximum sequence length: 256

Average tokens per text: 25.86
Maximum tokens: 186
Texts exceeding maximum length: 0


Batches:   0%|          | 0/165 [00:00<?, ?it/s]

Batches:   0%|          | 0/71 [00:00<?, ?it/s]


MiniLM training shape: (5261, 384)
MiniLM test shape: (2255, 384)


In [8]:
X_train_minilm

array([[ 0.02324069,  0.03298893,  0.07584879, ...,  0.04182152,
        -0.0784472 ,  0.0113375 ],
       [-0.04939846,  0.04506461,  0.02483919, ..., -0.04954027,
        -0.03933519,  0.00147051],
       [ 0.01776193,  0.13223131,  0.05443739, ..., -0.05853306,
        -0.12931313,  0.03321038],
       ...,
       [-0.04907256,  0.06152787,  0.00114814, ...,  0.06453512,
        -0.08033057,  0.00876078],
       [-0.01972462,  0.01427154,  0.05384759, ..., -0.0470631 ,
        -0.07787871,  0.03353288],
       [ 0.01166539,  0.10670753, -0.00268219, ...,  0.05720961,
        -0.15007664,  0.01010651]], shape=(5261, 384), dtype=float32)

## Cross-validation

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy": "accuracy", "macro_f1": "f1_macro"}

models = {"LR - BoW": Pipeline([("vectorizer", CountVectorizer(max_features=2000, min_df=2)),
        ("classifier", LogisticRegression(max_iter=2000))]),
    "LR - TF-IDF": Pipeline([("vectorizer", TfidfVectorizer(max_features=2000, min_df=2)),
        ("classifier", LogisticRegression(max_iter=2000))]),
    "LR - Word2Vec": Pipeline([("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000))]),
    "LR - MiniLM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000))])}


# cross-validation:
experiments = {}

for name, model in models.items():

    if "Word2Vec" in name:
        features = X_train_word2vec
    elif "MiniLM" in name:
        features = X_train_minilm
    else:
        features = X_train

    experiments[name] = cross_validate(model, features, y_train,
                                       cv=cv, scoring=scoring)

    print("Finished:", name)

Finished: LR - BoW
Finished: LR - TF-IDF
Finished: LR - Word2Vec
Finished: LR - MiniLM


In [10]:
results = []

for name, scores in experiments.items():
    results.append({"Model": name,
        "Mean accuracy": scores["test_accuracy"].mean(),
        "Std accuracy": scores["test_accuracy"].std(),
        "Mean Macro F1": scores["test_macro_f1"].mean(),
        "Std Macro F1": scores["test_macro_f1"].std()})

results_df = pd.DataFrame(results).sort_values("Mean accuracy", ascending=False)

print(results_df.round(4).to_string(index=False))

        Model  Mean accuracy  Std accuracy  Mean Macro F1  Std Macro F1
  LR - MiniLM         0.5776        0.0099         0.5755        0.0101
  LR - TF-IDF         0.5697        0.0090         0.5675        0.0101
     LR - BoW         0.5531        0.0129         0.5508        0.0141
LR - Word2Vec         0.5364        0.0188         0.5337        0.0184


# 2: Explore what your chosen word embedding model captures. 
Some things you might consider:
- Is it good at representing interesting similarities (re to your choice of task)? J
- Have you found task-specific evidence that it's better than some other reasonable alternative?
- Do you know about limitations of this model?
- Any limitations that you find particularly troubling for your task?

In [11]:
same_emotion = []
different_emotion = []

# generate normalized embeddings
embeddings = minilm.encode(X_train, convert_to_numpy=True, 
                           normalize_embeddings=True, show_progress_bar=True)

random.seed(42)
for _ in range(2000):
    i, j = random.sample(range(len(X_train)), 2)
    similarity = np.dot(embeddings[i], embeddings[j])

    if y_train[i] == y_train[j]:
        same_emotion.append(similarity)
    else:
        different_emotion.append(similarity)

print("Average similarity for same emotion:",
      round(np.mean(same_emotion), 4))

print("Average similarity for different emotions:",
      round(np.mean(different_emotion), 4))

Batches:   0%|          | 0/165 [00:00<?, ?it/s]

Average similarity for same emotion: 0.2467
Average similarity for different emotions: 0.2062


In [12]:
# select a specific sentence
query_index = 3

# calculate similarity with all training sentences
similarities = cosine_similarity(
    embeddings[query_index].reshape(1, -1),
    embeddings
)[0]

# five most similar sentences
similar_indices = np.argsort(similarities)[::-1]
similar_indices = [i for i in similar_indices if i != query_index][:5]

print("Original sentence:")
print(X_train[query_index])
print("Emotion:", y_train[query_index])

print("\nMost similar sentences:")

for i in similar_indices:
    print(f"\nSimilarity: {similarities[i]:.3f}")
    print("Sentence:", X_train[i])
    print("Emotion:", y_train[i])

Original sentence:
The day after the night I drank too much at a party.
Emotion: shame

Most similar sentences:

Similarity: 0.724
Sentence: On New Years eve I drank too much alcohol, so much that I had to  
vomit in the presence of other people.
Emotion: shame

Similarity: 0.698
Sentence: The day I came home drunk after having attended a disco party.
Emotion: guilt

Similarity: 0.695
Sentence: We had had a party the night before and we still had some liquors  
left the morning after. I went to the kitchen pretending to  
prepare some tea and was caught by my fiancee pouring some rum  
down my throat.
Emotion: guilt

Similarity: 0.688
Sentence: In a party of friends I got completely drunk.
Emotion: shame

Similarity: 0.684
Sentence: When I had drunk too much and nearly got an alcohol poisoning.
Emotion: guilt


In [15]:
print(model)
print(model.named_steps.keys())

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', LogisticRegression(max_iter=2000))])
dict_keys(['scaler', 'classifier'])


In [18]:
print("Maximum sequence length:", minilm.max_seq_length)

token_lengths = [len(minilm.tokenizer.tokenize(text)) for text in X_train]

print("Longest text:", max(token_lengths))
print("Average token length:", np.mean(token_lengths))

print("Texts exceeding maximum length:",
    sum(length > minilm.max_seq_length for length in token_lengths))

Maximum sequence length: 256
Longest text: 186
Average token length: 25.86162326553887
Texts exceeding maximum length: 0


In [14]:
# predict emotions using cross-validation
minilm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

oof_predictions = cross_val_predict(minilm_pipeline, embeddings, y_train, cv=cv)

# select two random sentences per emotion
np.random.seed(42)
selected_indices = []

for emotion in np.unique(y_train):
    emotion_indices = np.where(np.array(y_train) == emotion)[0]
    selected = np.random.choice(emotion_indices, size=2, replace=False)
    selected_indices.extend(selected)

# find the five most similar sentences for each selected sentence
similarity_matrix = cosine_similarity(embeddings)
results = []

for query_index in selected_indices:
    nearest_indices = np.argsort(similarity_matrix[query_index])[::-1]
    nearest_indices = [i for i in nearest_indices if i != query_index][:5]

    print("\n" + "=" * 60)
    print("Original sentence:", X_train[query_index])
    print("Actual emotion:", y_train[query_index])
    print("Predicted emotion:", oof_predictions[query_index])
    print("Correct prediction:", oof_predictions[query_index] == y_train[query_index])

    print("\nMost similar sentences:")

    for neighbour_index in nearest_indices:
        similarity = similarity_matrix[query_index, neighbour_index]
        actual = y_train[neighbour_index]
        predicted = oof_predictions[neighbour_index]

        print(f"\nSimilarity: {similarity:.3f}")
        print("Sentence:", X_train[neighbour_index])
        print("Actual emotion:", actual)
        print("Predicted emotion:", predicted)

        results.append({"Original emotion": y_train[query_index],
            "Original prediction": oof_predictions[query_index],
            "Neighbour emotion": actual,
            "Neighbour prediction": predicted,
            "Similarity": similarity,
            "Correct prediction": predicted == actual,
            "Same emotion": actual == y_train[query_index]})


# summarise the results
results_df = pd.DataFrame(results)

print("\nAverage similarity:", round(results_df["Similarity"].mean(), 3))
print("Neighbours with the same emotion:", round(results_df["Same emotion"].mean() * 100, 2), "%")
print("Correct neighbour predictions:", round(results_df["Correct prediction"].mean() * 100, 2), "%")

display(results_df.round(3))


Original sentence: My parents thought that I didn't do my best to pass the final  
examination (secondary school). When I achieved a good grade they  
said nothing, but if I had had a bad grade they would have become  
very angry.
Actual emotion: anger
Predicted emotion: anger
Correct prediction: True

Most similar sentences:

Similarity: 0.755
Sentence: Telling my parents that I did not well in an examn.
Actual emotion: fear
Predicted emotion: shame

Similarity: 0.689
Sentence: I had not told my parents about my bad marks in school. One day,  
my father wanted to speak to the teachers. Fortunately I could  
influence my parents to take it easy.
Actual emotion: fear
Predicted emotion: shame

Similarity: 0.686
Sentence: When I get an exam back with a grade that I did not desire. Then  
I get mad at myself and say that I could have done better.
Actual emotion: shame
Predicted emotion: anger

Similarity: 0.646
Sentence: I felt anger at my last exam, I felt angry with our teacher,  
becau

,Original emotion,Original prediction,Neighbour emotion,Neighbour prediction,Similarity,Correct prediction,Same emotion
0,anger,anger,fear,shame,0.755,False,False
1,anger,anger,fear,shame,0.689,False,False
2,anger,anger,shame,anger,0.686,False,False
3,anger,anger,anger,anger,0.646,True,True
4,anger,anger,guilt,guilt,0.633,True,False
...,...,...,...,...,...,...,...
65,shame,guilt,sadness,shame,0.751,False,False
66,shame,guilt,anger,fear,0.710,False,False
67,shame,guilt,shame,sadness,0.645,False,True
68,shame,guilt,shame,shame,0.497,True,True


# 3. Experiment with the embed-and-pool approach.

- What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
- Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?

We experimented with two pretrained embedding approaches: Word2Vec and Sentence-BERT (MiniLM). Word2Vec uses 300-dimensional static word embeddings, which we combined using mean pooling to create sentence representations. MiniLM generates contextual sentence embeddings of 384 dimensions.

The results indicate that the choice of embedding architecture has an impact on classification performance. MiniLM achieved the highest accuracy (57.76%), whereas Word2Vec achieved only 53.64%. This suggests that contextual sentence embeddings are more suitable for this emotion classification task than averaging static word embeddings. Compared with the traditional TF-IDF approach (56.97%), MiniLM provided a small improvement of approximately 0.79 percentage points. Therefore, pretrained embeddings added some value to the model, although the improvement was limited. Word2Vec did not improve performance compared with the traditional approaches.

Overall, the experiments suggest that the quality and contextual nature of the embeddings are more important than simply using pretrained word vectors.

# 4. Experiment with a recurrent encoder (GRU or LSTM).

- What design choices have meaningful impact in classification performance, if any?
- Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
- Did you struggle with over-fitting?
- How far can this approach go? Are you able to improve meaningfully on embed-and-pool?

In [19]:
from transformers import AutoTokenizer, AutoModel
import torch

class PretrainedEmbeddings:
    """
    Load an embedding matrix (stored in one of the supported files, see read_lines_from_file)
     and gives structured access to it.
    """

    def __init__(self, model_name, training_data, num_dims, unk_token="[UNK]", pad_token="[PAD]", encoding="utf-8", skip_first_line=False):
        """
        path: text/gz/zip file with embeddings
        num_dims: dimensionality of embeddings
        unk_token: special token reserve for words not in the table, we use the average of all embeddings as the unk embedding
        pad_token: special token used for batching sequences of variable length, we use a vector of zeros
        encoding: encoding of the embeddings file
        skip_first_line: some embedding files have metadata in the first line
        """
        vectors = [
            np.zeros(num_dims, dtype=np.float32),  # pad
            np.zeros(num_dims, dtype=np.float32) # unk placeholder
        ]
        self.vocab = part2.clslib.Vocabulary(init=[pad_token, unk_token], default=unk_token)
        self.num_dims = num_dims
        self.unk_token = unk_token
        self.pad_token = pad_token

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)

        for sentence in training_data: 
            
            inputs = tokenizer(sentence, padding=False, truncation=True, return_tensors="pt")
    
            with torch.no_grad():
                outputs = model(**inputs)
            
            token_embeddings = outputs.last_hidden_state
            
            tokens = tokenizer.convert_ids_to_tokens(
                inputs["input_ids"][0]
            )
    
            for j, word in enumerate(tokens):
        
                # Skip padding and special tokens
                if word in tokenizer.all_special_tokens:
                    continue
        
                # Skip words already in vocabulary
                if word in self.vocab:
                    continue

                self.vocab.add(word)

                # Store corresponding embedding
                vectors.append(token_embeddings[0, j].numpy())

        # [num_words, num_dims]
        vectors = np.stack(vectors, axis=0)
        # unk embedding is the mean of all others after pad (that's the "center" of the space)
        vectors[self.vocab.default_index] = np.mean(vectors[2:, :], 0)
        self.vectors = vectors
        # we freeze the vocab to avoid having it out of sync with the vectors
        self.vocab.freeze()
        assert self.vectors.shape[0] == len(self.vocab), "Mismatch between vocab and vectors"

    def __contains__(self, word: str):
        """Test whether or not a word (or its lowercased version) is in the vocabulary"""
        return word.lower() in self.vocab

    def index(self, word: str):
        """Return the id of a word (or of its lowercased version) or 0 (by definition, id of unk_token)"""
        return self.vocab.index(word.lower())

    def word(self, i: int):
        """Return the word associated with an id"""
        return self.vocab.word(i)

    def __getitem__(self, i: int):
        """Return self.vectors[i]"""
        return self.vectors[i]  # this will raise an exceptino if i is out of bounds

    def embed_word(self, word: str):
        """Return the vector representation of the word (or its lowercased version, or unk)"""
        i = self.index(word)
        return self.vectors[i]

    def embed_sequence(self, tokens: list):
        """Return a list of embeddings, one for each token in the input list"""
        return [self.embed_word(t) for t in tokens]


In [20]:
minilm_emb = PretrainedEmbeddings(model_name, X_train, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
tokenizer_for_rnn = part2.clslib.BasicTokenizer(
    lowercase=True,
    remove_punct=False,
    remove_stopwords=False,
    language='english',
    vocab=minilm_emb.vocab
)

In [22]:
my_seeds = [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
cross_validation_set = []
outcome_space = part1.clslib.NamedClasses(["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"])

for seed in my_seeds:  
    cross_validation_set.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(X, np.array([outcome_space.index(x) for x in y]))),  
            k=num_folds,
            seed=seed
        )
    )

In [23]:
hyperparameters = {
    "hidden_size" : [32, 64, 128, 256],
    "bidirectional" : [False, True],
    "freeze_embeddings" : [True, False],
    "pretrained_embeddings": [True, False],
    "word_dropout" : [0.0, 0.1, 0.2, 0.3]
}

In [26]:
import nltk
nltk.download('punkt_tab')
my_classifier = {}
my_classifier['Bi=False'] = []

for run, seed, in enumerate(my_seeds):
    run_folder = f"results/lstm/Bi=False-seed-{seed}"
    my_classifier['Bi=False'].append(
        part2.experiment.cross_validation(
            outcome_space=outcome_space,
            data_splits=cross_validation_set[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=minilm_emb,
            enc_type='lstm', 
            enc_kwargs={'bidirectional': False, 'hidden_size': 128},
            regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 10},    
            device='cpu', # 'cuda' or 'cpu'
            save_to=run_folder,        
        )
    )



[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\juliu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.761 seconds
Dataloader done! Time taken: 0.778 seconds


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7597 | train acc: 0.2868 | val loss: 1.5185 | val acc: 0.4242
Epoch 02 | train loss: 1.4006 | train acc: 0.4800 | val loss: 1.3769 | val acc: 0.5000
Epoch 03 | train loss: 1.2130 | train acc: 0.5679 | val loss: 1.2550 | val acc: 0.5479
Epoch 04 | train loss: 1.0783 | train acc: 0.6248 | val loss: 1.2439 | val acc: 0.5485
Epoch 05 | train loss: 0.9636 | train acc: 0.6702 | val loss: 1.2276 | val acc: 0.5559
Epoch 06 | train loss: 0.8610 | train acc: 0.7071 | val loss: 1.2791 | val acc: 0.5652
Epoch 07 | train loss: 0.7671 | train acc: 0.7389 | val loss: 1.2744 | val acc: 0.5725
Epoch 08 | train loss: 0.6803 | train acc: 0.7759 | val loss: 1.3813 | val acc: 0.5585
Epoch 09 | train loss: 0.5806 | train acc: 0.8137 | val loss: 1.4007 | val acc: 0.5765
Epoch 10 | train loss: 0.4939 | train acc: 0.8398 | val loss: 1.4505 | val acc: 0.5632
Training done! Time taken: 467.756 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.708 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

... (27 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7725 | train acc: 0.2827 | val loss: 1.5054 | val acc: 0.4478
Epoch 02 | train loss: 1.4268 | train acc: 0.4696 | val loss: 1.3054 | val acc: 0.5243
Epoch 03 | train loss: 1.2291 | train acc: 0.5588 | val loss: 1.2364 | val acc: 0.5529
Epoch 04 | train loss: 1.0979 | train acc: 0.6172 | val loss: 1.1408 | val acc: 0.5955
Epoch 05 | train loss: 0.9648 | train acc: 0.6684 | val loss: 1.1826 | val acc: 0.5788
Epoch 06 | train loss: 0.8902 | train acc: 0.6922 | val loss: 1.1525 | val acc: 0.5928
Epoch 07 | train loss: 0.7783 | train acc: 0.7359 | val loss: 1.1740 | val acc: 0.5988
Epoch 08 | train loss: 0.6897 | train acc: 0.7672 | val loss: 1.2269 | val acc: 0.6075
Epoch 09 | train loss: 0.5786 | train acc: 0.8031 | val loss: 1.3393 | val acc: 0.5981
Epoch 10 | train loss: 0.5090 | train acc: 0.8299 | val loss: 1.3838 | val acc: 0.6001
Training done! Time taken: 330.449 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.332 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7557 | train acc: 0.2965 | val loss: 1.5593 | val acc: 0.4032
Epoch 02 | train loss: 1.3813 | train acc: 0.5011 | val loss: 1.3975 | val acc: 0.4877
Epoch 03 | train loss: 1.1984 | train acc: 0.5688 | val loss: 1.3407 | val acc: 0.5210
Epoch 04 | train loss: 1.0715 | train acc: 0.6273 | val loss: 1.2654 | val acc: 0.5476
Epoch 05 | train loss: 0.9598 | train acc: 0.6717 | val loss: 1.2819 | val acc: 0.5576
Epoch 06 | train loss: 0.8706 | train acc: 0.7026 | val loss: 1.3012 | val acc: 0.5556
Epoch 07 | train loss: 0.7782 | train acc: 0.7379 | val loss: 1.3552 | val acc: 0.5655
Epoch 08 | train loss: 0.6686 | train acc: 0.7773 | val loss: 1.4229 | val acc: 0.5549
Epoch 09 | train loss: 0.5915 | train acc: 0.8029 | val loss: 1.4314 | val acc: 0.5595
Epoch 10 | train loss: 0.5006 | train acc: 0.8362 | val loss: 1.5403 | val acc: 0.5615
Training done! Time taken: 322.290 seconds


In [29]:
my_classifier['Bi=True'] = []

for run, seed, in enumerate(my_seeds):
    run_folder = f"results/lstm/Bi=True-seed-{seed}"
    my_classifier['Bi=True'].append(
        part2.experiment.cross_validation(
            outcome_space=outcome_space,
            data_splits=cross_validation_set[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=minilm_emb,
            enc_type='lstm', 
            enc_kwargs={'bidirectional': True, 'hidden_size': 64},
            regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 10},    
            device='cpu', # "cuda" or'cpu'
            save_to=run_folder,        
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.399 seconds
Dataloader done! Time taken: 0.139 seconds


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7673 | train acc: 0.3022 | val loss: 1.4753 | val acc: 0.4422
Epoch 02 | train loss: 1.3216 | train acc: 0.5200 | val loss: 1.2670 | val acc: 0.5259
Epoch 03 | train loss: 1.0854 | train acc: 0.6158 | val loss: 1.2035 | val acc: 0.5592
Epoch 04 | train loss: 0.9273 | train acc: 0.6765 | val loss: 1.1776 | val acc: 0.5711
Epoch 05 | train loss: 0.7785 | train acc: 0.7239 | val loss: 1.2564 | val acc: 0.5765
Epoch 06 | train loss: 0.6840 | train acc: 0.7623 | val loss: 1.2726 | val acc: 0.5864
Epoch 07 | train loss: 0.5671 | train acc: 0.8044 | val loss: 1.3545 | val acc: 0.5824
Epoch 08 | train loss: 0.4578 | train acc: 0.8505 | val loss: 1.4334 | val acc: 0.5705
Epoch 09 | train loss: 0.3701 | train acc: 0.8794 | val loss: 1.5180 | val acc: 0.5758
Epoch 10 | train loss: 0.2879 | train acc: 0.9102 | val loss: 1.6295 | val acc: 0.5625
Training done! Time taken: 412.550 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.324 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7766 | train acc: 0.3050 | val loss: 1.4878 | val acc: 0.4604
Epoch 02 | train loss: 1.2940 | train acc: 0.5322 | val loss: 1.2601 | val acc: 0.5549
Epoch 03 | train loss: 1.0667 | train acc: 0.6172 | val loss: 1.1841 | val acc: 0.5775
Epoch 04 | train loss: 0.9083 | train acc: 0.6834 | val loss: 1.1695 | val acc: 0.5941
Epoch 05 | train loss: 0.7847 | train acc: 0.7316 | val loss: 1.1994 | val acc: 0.6075
Epoch 06 | train loss: 0.6617 | train acc: 0.7831 | val loss: 1.2986 | val acc: 0.5862
Epoch 07 | train loss: 0.5611 | train acc: 0.8122 | val loss: 1.3626 | val acc: 0.5775
Epoch 08 | train loss: 0.4650 | train acc: 0.8482 | val loss: 1.4597 | val acc: 0.5875
Epoch 09 | train loss: 0.3770 | train acc: 0.8763 | val loss: 1.5262 | val acc: 0.5882
Epoch 10 | train loss: 0.3070 | train acc: 0.9020 | val loss: 1.6600 | val acc: 0.5722
Training done! Time taken: 407.891 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.291 seconds
Dataloader done! Time 

... (26 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7538 | train acc: 0.3077 | val loss: 1.5252 | val acc: 0.4178
Epoch 02 | train loss: 1.3486 | train acc: 0.5044 | val loss: 1.2463 | val acc: 0.5329
Epoch 03 | train loss: 1.1334 | train acc: 0.5939 | val loss: 1.1833 | val acc: 0.5709
Epoch 04 | train loss: 0.9645 | train acc: 0.6676 | val loss: 1.1318 | val acc: 0.5981
Epoch 05 | train loss: 0.8235 | train acc: 0.7189 | val loss: 1.2007 | val acc: 0.6094
Epoch 06 | train loss: 0.7001 | train acc: 0.7627 | val loss: 1.1534 | val acc: 0.6154
Epoch 07 | train loss: 0.5891 | train acc: 0.8029 | val loss: 1.2167 | val acc: 0.6075
Epoch 08 | train loss: 0.4620 | train acc: 0.8495 | val loss: 1.3487 | val acc: 0.5875
Epoch 09 | train loss: 0.3722 | train acc: 0.8806 | val loss: 1.4178 | val acc: 0.5888
Epoch 10 | train loss: 0.2872 | train acc: 0.9110 | val loss: 1.5139 | val acc: 0.6041
Training done! Time taken: 409.004 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.331 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7454 | train acc: 0.3228 | val loss: 1.5357 | val acc: 0.4245
Epoch 02 | train loss: 1.3372 | train acc: 0.5082 | val loss: 1.3429 | val acc: 0.5070
Epoch 03 | train loss: 1.1123 | train acc: 0.6054 | val loss: 1.3469 | val acc: 0.5356
Epoch 04 | train loss: 0.9663 | train acc: 0.6700 | val loss: 1.2215 | val acc: 0.5722
Epoch 05 | train loss: 0.8347 | train acc: 0.7105 | val loss: 1.2468 | val acc: 0.5755
Epoch 06 | train loss: 0.6937 | train acc: 0.7647 | val loss: 1.2914 | val acc: 0.5749
Epoch 07 | train loss: 0.5776 | train acc: 0.8121 | val loss: 1.3129 | val acc: 0.5782
Epoch 08 | train loss: 0.4736 | train acc: 0.8415 | val loss: 1.4684 | val acc: 0.5702
Epoch 09 | train loss: 0.3766 | train acc: 0.8828 | val loss: 1.5725 | val acc: 0.5576
Epoch 10 | train loss: 0.2972 | train acc: 0.9064 | val loss: 1.6679 | val acc: 0.5509
Training done! Time taken: 408.636 seconds


In [30]:
df1 = part1.analysis.make_data_frame(my_classifier)
means_bi = []
stds_bi = []
for i in range(0, 30, 5):
    acc = np.array(df1['accuracy'][i:i+5])
    means_bi.append(np.mean(acc))
    stds_bi.append(np.std(acc))

print("Unidirectional:")
print("Inter-fold Means:", [float(x) for x in means_bi[:3]])
print("Inter-fold Standard deviations:", [float(x) for x in stds_bi[:3]])
print("Inter-seed Mean:", np.mean(np.array(means_bi[:3])))
print("Inter-seed Standard deviation:", np.std(np.array(means_bi[:3])))

print("\nBidirectional:")
print("Inter-fold Means:", [float(x) for x in means_bi[3:]])
print("Inter-fold Standard deviations:", [float(x) for x in stds_bi[3:]])
print("Inter-seed Mean:", np.mean(np.array(means_bi[3:])))
print("Inter-seed Standard Deviation:", np.std(np.array(means_bi[3:])))

Unidirectional:
Inter-fold Means: [0.5745101994592375, 0.5864813812092128, 0.5871474250081398]
Inter-fold Standard deviations: [0.01257513246575015, 0.011999471693908957, 0.020770860711903726]
Inter-seed Mean: 0.5827130018921967
Inter-seed Standard deviation: 0.005806627213861922

Bidirectional:
Inter-fold Means: [0.5850204732379213, 0.5971262262708625, 0.5948639069378973]
Inter-fold Standard deviations: [0.008087397112858668, 0.008395898379364889, 0.013947789769597353]
Inter-seed Mean: 0.5923368688155604
Inter-seed Standard Deviation: 0.005255267540125787


In [33]:
my_seeds = [0]
for dim in hyperparameters['hidden_size']:
    my_classifier[dim] = []
    
    for run, seed, in enumerate(my_seeds):
        run_folder = f"results/lstm/dim-{dim}-seed-{seed}"
        my_classifier[dim].append(
            part2.experiment.cross_validation(
                outcome_space=outcome_space,
                data_splits=cross_validation_set[run],
                tokenizer=tokenizer_for_rnn,
                emb_model=minilm_emb,
                enc_type='lstm', 
                enc_kwargs={'bidirectional': False, 'hidden_size': dim},
                regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
                opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 10},    
                device='cpu', #'cpu'
                save_to=run_folder,        
            )
        )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.448 seconds
Dataloader done! Time taken: 0.113 seconds


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8973 | train acc: 0.2058 | val loss: 1.7075 | val acc: 0.3384
Epoch 02 | train loss: 1.5572 | train acc: 0.4271 | val loss: 1.4455 | val acc: 0.4787
Epoch 03 | train loss: 1.3348 | train acc: 0.5185 | val loss: 1.3530 | val acc: 0.5093
Epoch 04 | train loss: 1.1890 | train acc: 0.5788 | val loss: 1.2747 | val acc: 0.5479
Epoch 05 | train loss: 1.0897 | train acc: 0.6229 | val loss: 1.2816 | val acc: 0.5366
Epoch 06 | train loss: 1.0095 | train acc: 0.6510 | val loss: 1.2860 | val acc: 0.5539
Epoch 07 | train loss: 0.9359 | train acc: 0.6836 | val loss: 1.2857 | val acc: 0.5559
Epoch 08 | train loss: 0.8725 | train acc: 0.7021 | val loss: 1.2850 | val acc: 0.5638
Epoch 09 | train loss: 0.8013 | train acc: 0.7267 | val loss: 1.3555 | val acc: 0.5552
Epoch 10 | train loss: 0.7580 | train acc: 0.7418 | val loss: 1.3363 | val acc: 0.5751
Training done! Time taken: 158.348 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.947 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8769 | train acc: 0.2303 | val loss: 1.7602 | val acc: 0.3074
Epoch 02 | train loss: 1.5558 | train acc: 0.4126 | val loss: 1.4560 | val acc: 0.4604
Epoch 03 | train loss: 1.3463 | train acc: 0.4981 | val loss: 1.3865 | val acc: 0.5070
Epoch 04 | train loss: 1.2038 | train acc: 0.5699 | val loss: 1.3063 | val acc: 0.5323
Epoch 05 | train loss: 1.0975 | train acc: 0.6055 | val loss: 1.2779 | val acc: 0.5615
Epoch 06 | train loss: 1.0210 | train acc: 0.6451 | val loss: 1.2914 | val acc: 0.5729
Epoch 07 | train loss: 0.9408 | train acc: 0.6719 | val loss: 1.3393 | val acc: 0.5622
Epoch 08 | train loss: 0.8771 | train acc: 0.6985 | val loss: 1.3295 | val acc: 0.5615
Epoch 09 | train loss: 0.8164 | train acc: 0.7223 | val loss: 1.3689 | val acc: 0.5689
Epoch 10 | train loss: 0.7775 | train acc: 0.7312 | val loss: 1.3400 | val acc: 0.5709
Training done! Time taken: 152.904 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.213 seconds
Dataloader done! Time 

... (38 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7539 | train acc: 0.3027 | val loss: 1.5749 | val acc: 0.3899
Epoch 02 | train loss: 1.4712 | train acc: 0.4437 | val loss: 1.4970 | val acc: 0.4192
Epoch 03 | train loss: 1.3051 | train acc: 0.5160 | val loss: 1.3060 | val acc: 0.5203
Epoch 04 | train loss: 1.1752 | train acc: 0.5846 | val loss: 1.2813 | val acc: 0.5436
Epoch 05 | train loss: 1.0425 | train acc: 0.6315 | val loss: 1.2088 | val acc: 0.5855
Epoch 06 | train loss: 0.9225 | train acc: 0.6835 | val loss: 1.2407 | val acc: 0.5709
Epoch 07 | train loss: 0.8365 | train acc: 0.7141 | val loss: 1.3020 | val acc: 0.5782
Epoch 08 | train loss: 0.7237 | train acc: 0.7547 | val loss: 1.2628 | val acc: 0.5842
Epoch 09 | train loss: 0.6123 | train acc: 0.7951 | val loss: 1.3955 | val acc: 0.5729
Epoch 10 | train loss: 0.4987 | train acc: 0.8315 | val loss: 1.4888 | val acc: 0.5862
Training done! Time taken: 922.791 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.264 seconds
Dataloader done! Time 

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7314 | train acc: 0.3236 | val loss: 1.4561 | val acc: 0.4617
Epoch 02 | train loss: 1.4317 | train acc: 0.4690 | val loss: 1.3854 | val acc: 0.5057
Epoch 03 | train loss: 1.2588 | train acc: 0.5466 | val loss: 1.2534 | val acc: 0.5529
Epoch 04 | train loss: 1.1148 | train acc: 0.6207 | val loss: 1.1965 | val acc: 0.5755
Epoch 05 | train loss: 1.0303 | train acc: 0.6473 | val loss: 1.2113 | val acc: 0.5702
Epoch 06 | train loss: 0.9735 | train acc: 0.6659 | val loss: 1.1576 | val acc: 0.5921
Epoch 07 | train loss: 0.8256 | train acc: 0.7189 | val loss: 1.1777 | val acc: 0.5902
Epoch 08 | train loss: 0.7231 | train acc: 0.7570 | val loss: 1.2631 | val acc: 0.5862
Epoch 09 | train loss: 0.6281 | train acc: 0.7900 | val loss: 1.3243 | val acc: 0.6008
Epoch 10 | train loss: 0.5167 | train acc: 0.8362 | val loss: 1.3449 | val acc: 0.6028
Training done! Time taken: 919.116 seconds


In [34]:
df2 = part1.analysis.make_data_frame(my_classifier)
means_hidden = []
stds_hidden = []

for i in range(30, 50, 5):
    acc = np.array(df2['accuracy'][i:i+5])
    means_hidden.append(np.mean(acc))
    stds_hidden.append(np.std(acc))
    
print("Hidden size 32:")
print("Mean:", means_hidden[1])
print("Standard deviation:", stds_hidden[1])

print("\nHidden size 64:")
print("Mean:", means_hidden[2])
print("Standard deviation:", stds_hidden[2])

print("\nHidden size 128:")
print("Mean:", means_hidden[0])
print("Standard deviation:", stds_hidden[0])

print("\nHidden size 256:")
print("Mean:", means_hidden[3])
print("Standard deviation:", stds_hidden[3])

Hidden size 32:
Mean: 0.5771687122209481
Standard deviation: 0.008357874172974601

Hidden size 64:
Mean: 0.5894086826347306
Standard deviation: 0.012901239054190406

Hidden size 128:
Mean: 0.5629344148582268
Standard deviation: 0.009167961424328675

Hidden size 256:
Mean: 0.5879483939921576
Standard deviation: 0.012133583360208216


In [38]:
# had to change num_epochs to 5 instead of 20 due to lang computing times 

for comb in [(True, True), (True, False), (False,False)]:
    my_classifier[comb] = []

    for run, seed, in enumerate(my_seeds):
        run_folder = f"results/lstm/pretrained-{comb[0]}-frozen-{comb[1]}-seed-{seed}"
        my_classifier[comb].append(
            part2.experiment.cross_validation(
                outcome_space=outcome_space,
                data_splits=cross_validation_set[run],
                tokenizer=tokenizer_for_rnn,
                emb_model=minilm_emb,
                enc_type='lstm', 
                enc_kwargs={'bidirectional': False, 'hidden_size': 128, 'pretrained':comb[0], 'freeze_embeddings':comb[1]},
                regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
                opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 5},    
                device='cpu',
                save_to=run_folder,        
            )
        )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.281 seconds
Dataloader done! Time taken: 0.131 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7797 | train acc: 0.2813 | val loss: 1.5001 | val acc: 0.4089
Epoch 02 | train loss: 1.3925 | train acc: 0.4862 | val loss: 1.3227 | val acc: 0.5273
Epoch 03 | train loss: 1.1992 | train acc: 0.5727 | val loss: 1.3099 | val acc: 0.5093
Epoch 04 | train loss: 1.0678 | train acc: 0.6257 | val loss: 1.2140 | val acc: 0.5698
Epoch 05 | train loss: 0.9665 | train acc: 0.6678 | val loss: 1.2018 | val acc: 0.5798
Training done! Time taken: 168.658 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.350 seconds
Dataloader done! Time taken: 0.111 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7495 | train acc: 0.3043 | val loss: 1.4966 | val acc: 0.4551
Epoch 02 | train loss: 1.3818 | train acc: 0.4969 | val loss: 1.2958 | val acc: 0.5296
Epoch 03 | train loss: 1.1933 | train acc: 0.5796 | val loss: 1.2734 | val acc: 0.5609
Epoch 04 | train loss: 1.0984 | train acc: 0.6177 | val loss: 1.2204 | val acc: 0.5828
Epoch 05 | train loss: 0.9923 | train acc: 0.6597 | val loss: 1.2446 | val acc: 0.5682
Training done! Time taken: 171.821 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.521 seconds
Dataloader done! Time taken: 0.103 seconds


... (26 outputs left out)

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9037 | train acc: 0.2155 | val loss: 1.7938 | val acc: 0.3167
Epoch 02 | train loss: 1.5358 | train acc: 0.4304 | val loss: 1.5171 | val acc: 0.4484
Epoch 03 | train loss: 1.1018 | train acc: 0.6138 | val loss: 1.5132 | val acc: 0.4777
Epoch 04 | train loss: 0.7456 | train acc: 0.7475 | val loss: 1.5793 | val acc: 0.4917
Epoch 05 | train loss: 0.5051 | train acc: 0.8397 | val loss: 1.7825 | val acc: 0.4963
Training done! Time taken: 194.500 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.301 seconds
Dataloader done! Time taken: 0.123 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.9158 | train acc: 0.2029 | val loss: 1.8218 | val acc: 0.2888
Epoch 02 | train loss: 1.5990 | train acc: 0.4116 | val loss: 1.5516 | val acc: 0.4165
Epoch 03 | train loss: 1.1248 | train acc: 0.6080 | val loss: 1.5278 | val acc: 0.4424
Epoch 04 | train loss: 0.7527 | train acc: 0.7500 | val loss: 1.5964 | val acc: 0.4990
Epoch 05 | train loss: 0.4906 | train acc: 0.8408 | val loss: 1.7214 | val acc: 0.4904
Training done! Time taken: 196.492 seconds


In [39]:
df3 = part1.analysis.make_data_frame(my_classifier)
    
print("Pretrained=True, Freeze_embeddings=True:")
print("Mean:", means_hidden[0])
print("Standard deviation:", stds_hidden[0])

print("\nPretrained=True, Freeze_embeddings=False:")
print("Mean:", np.mean(np.array(df3['accuracy'][50:55])))
print("Standard deviation:", np.std(np.array(df3['accuracy'][50:55])))

print("\nPretrained=False, Freeze_embeddings=False:")
print("Mean:", np.mean(np.array(df3['accuracy'][55:60])))
print("Standard deviation:", np.std(np.array(df3['accuracy'][55:60])))

Pretrained=True, Freeze_embeddings=True:
Mean: 0.5629344148582268
Standard deviation: 0.009167961424328675

Pretrained=True, Freeze_embeddings=False:
Mean: 0.575042733681573
Standard deviation: 0.01613416489564479

Pretrained=False, Freeze_embeddings=False:
Mean: 0.5859507934485639
Standard deviation: 0.006292778493889359


In [40]:
# had to change num_epochs to 5 instead of 20 due to lang computing times 

for drop in [0.1, 0.2, 0.3]:
    my_classifier[drop] = []

    for run, seed, in enumerate(my_seeds):
        run_folder = f"results/lstm/drop-{drop}-seed-{seed}"
        my_classifier[drop].append(
            part2.experiment.cross_validation(
                outcome_space=outcome_space,
                data_splits=cross_validation_set[run],
                tokenizer=tokenizer_for_rnn,
                emb_model=minilm_emb,
                enc_type='lstm', 
                enc_kwargs={'hidden_size': 128, 'word_dropout': drop},
                regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
                opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 5},    
                device='cpu',
                save_to=run_folder,        
            )
        )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 1.223 seconds
Dataloader done! Time taken: 0.115 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8324 | train acc: 0.2615 | val loss: 1.5808 | val acc: 0.4076
Epoch 02 | train loss: 1.5024 | train acc: 0.4484 | val loss: 1.3489 | val acc: 0.5126
Epoch 03 | train loss: 1.3167 | train acc: 0.5273 | val loss: 1.2995 | val acc: 0.5259
Epoch 04 | train loss: 1.1995 | train acc: 0.5752 | val loss: 1.2093 | val acc: 0.5658
Epoch 05 | train loss: 1.0784 | train acc: 0.6216 | val loss: 1.2216 | val acc: 0.5592
Training done! Time taken: 167.010 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.217 seconds
Dataloader done! Time taken: 0.110 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8004 | train acc: 0.2699 | val loss: 1.5474 | val acc: 0.4185
Epoch 02 | train loss: 1.4785 | train acc: 0.4505 | val loss: 1.3506 | val acc: 0.4943
Epoch 03 | train loss: 1.3028 | train acc: 0.5343 | val loss: 1.3300 | val acc: 0.5396
Epoch 04 | train loss: 1.1870 | train acc: 0.5728 | val loss: 1.2316 | val acc: 0.5509
Epoch 05 | train loss: 1.0915 | train acc: 0.6172 | val loss: 1.2236 | val acc: 0.5715
Training done! Time taken: 168.360 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.413 seconds
Dataloader done! Time taken: 0.116 seconds


... (26 outputs left out)

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8230 | train acc: 0.2579 | val loss: 1.6436 | val acc: 0.3693
Epoch 02 | train loss: 1.5529 | train acc: 0.4066 | val loss: 1.4433 | val acc: 0.4624
Epoch 03 | train loss: 1.4148 | train acc: 0.4818 | val loss: 1.3077 | val acc: 0.5356
Epoch 04 | train loss: 1.3499 | train acc: 0.5134 | val loss: 1.2781 | val acc: 0.5250
Epoch 05 | train loss: 1.2824 | train acc: 0.5403 | val loss: 1.2131 | val acc: 0.5729
Training done! Time taken: 170.594 seconds
Tokenizing texts...
Tokenization done! Time taken: 1.267 seconds
Dataloader done! Time taken: 0.111 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8334 | train acc: 0.2631 | val loss: 1.5275 | val acc: 0.4165
Epoch 02 | train loss: 1.5606 | train acc: 0.4124 | val loss: 1.3754 | val acc: 0.4970
Epoch 03 | train loss: 1.4337 | train acc: 0.4663 | val loss: 1.4106 | val acc: 0.4997
Epoch 04 | train loss: 1.3601 | train acc: 0.5117 | val loss: 1.2525 | val acc: 0.5529
Epoch 05 | train loss: 1.3042 | train acc: 0.5318 | val loss: 1.2058 | val acc: 0.5815
Training done! Time taken: 175.710 seconds


In [41]:
df4 = part1.analysis.make_data_frame(my_classifier)

print("No dropout")
print("Mean:", means_hidden[0])
print("Standard deviation:", stds_hidden[0])

print("\nDropout 0.1")
print("Mean:", np.mean(np.array(df4['accuracy'][60:65])))
print("Standard deviation:", np.std(np.array(df4['accuracy'][60:65])))

print("\nDropout 0.2")
print("Mean:", np.mean(np.array(df4['accuracy'][65:70])))
print("Standard deviation:", np.std(np.array(df4['accuracy'][65:70])))

print("\nDropout 0.3")
print("Mean:", np.mean(np.array(df4['accuracy'][70:75])))
print("Standard deviation:", np.std(np.array(df4['accuracy'][70:75])))

No dropout
Mean: 0.5629344148582268
Standard deviation: 0.009167961424328675

Dropout 0.1
Mean: 0.46953185826927707
Standard deviation: 0.014819708908524293

Dropout 0.2
Mean: 0.5808960978751716
Standard deviation: 0.011850374586974416

Dropout 0.3
Mean: 0.5729114466103254
Standard deviation: 0.00313077768016623
